# Lab 4 · Transformers Deep Dive — Multi-Head Attention from Scratch

> **Transformers teaching package — Lab.** Run top to bottom (Runtime → Run all). Read the notes, run the code,
> and finish the **Your turn 🧪** cell. Colors follow the *Neural Indigo* palette from the slides.


## What you'll build
The engine room of every LLM: **scaled dot-product attention**, then **multi-head** attention, then the three
mask types (padding, causal, cross). No `transformers` library — just tensors, so nothing is hidden.

**Concepts:** Q/K/V projections · scaled dot-product · multiple heads · causal & cross attention.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math, matplotlib.pyplot as plt
torch.manual_seed(0)
print("torch", torch.__version__)

## 1 · Scaled dot-product attention
For queries `Q`, keys `K`, values `V`:
$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$
The scaling by $\sqrt{d_k}$ keeps the dot products from growing with dimension and saturating the softmax.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)      # (..., q_len, k_len)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

# tiny sanity check
q = torch.randn(1, 4, 8); k = torch.randn(1, 6, 8); v = torch.randn(1, 6, 8)
out, w = scaled_dot_product_attention(q, k, v)
print("output:", out.shape, "| weights:", w.shape, "| each row sums to 1 ->", w.sum(-1).round())

## 2 · Multi-head attention
Instead of one attention, project into `h` heads, attend in each, concatenate, and mix with an output layer.
Each head can specialize (syntax, coreference, position…).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h, self.d_k = num_heads, d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split(self, x):                      # (B, L, d_model) -> (B, h, L, d_k)
        B, L, _ = x.shape
        return x.view(B, L, self.h, self.d_k).transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        Q, K, V = self.split(self.W_q(q)), self.split(self.W_k(k)), self.split(self.W_v(v))
        out, attn = scaled_dot_product_attention(Q, K, V, mask)   # (B, h, L, d_k)
        B, h, L, d_k = out.shape
        out = out.transpose(1, 2).contiguous().view(B, L, h * d_k)
        return self.W_o(out), attn

mha = MultiHeadAttention(d_model=32, num_heads=4)
x = torch.randn(1, 5, 32)
y, attn = mha(x, x, x)
print("input:", x.shape, "-> output:", y.shape, "| attn (B,h,L,L):", attn.shape)

## 3 · The three masks
- **Padding mask** — ignore `[PAD]` positions in a batch.
- **Causal mask** — a decoder position may not see the future (lower-triangular).
- **Cross-attention** — decoder *queries* attend to encoder *keys/values* (no mask needed, different sequences).

In [ ]:
def causal_mask(L):
    return torch.tril(torch.ones(L, L)).bool()      # 1 = keep, 0 = block

L = 6
m = causal_mask(L)
plt.figure(figsize=(4,4))
plt.imshow(m, cmap='Purples'); plt.title('Causal mask (lower triangle)')
plt.xlabel('key position'); plt.ylabel('query position'); plt.colorbar(); plt.show()

# apply it
scores = torch.randn(1,1,L,L)
masked = scores.masked_fill(~m, float('-inf'))
print("row 0 attends to:", (F.softmax(masked,dim=-1)[0,0,0] > 1e-6).int().tolist())
print("row 5 attends to:", (F.softmax(masked,dim=-1)[0,0,5] > 1e-6).int().tolist())

## 4 · Cross-attention
Encoder output = keys & values; decoder state = queries. Different lengths are fine.

In [ ]:
enc = torch.randn(1, 7, 32)   # 7 source tokens
dec = torch.randn(1, 4, 32)   # 4 target tokens so far
cross = MultiHeadAttention(32, 4)
out, attn = cross(dec, enc, enc)        # Q from decoder, K/V from encoder
print("decoder queries:", dec.shape[1], "| source keys:", enc.shape[1])
print("cross-attention shape (B,h,tgt,src):", attn.shape)

plt.figure(figsize=(5,3))
plt.imshow(attn[0].mean(0).detach(), cmap='viridis', aspect='auto')
plt.title('Cross-attention (avg over heads)'); plt.xlabel('source'); plt.ylabel('target'); plt.colorbar(); plt.show()

## Your turn 🧪
1. Change `num_heads` to 1 vs 8 — how does the attention pattern change?
2. Build a **full encoder block**: `MultiHeadAttention` → residual + `LayerNorm` → a 2-layer MLP (`d_model → 4*d_model → d_model`) → residual + `LayerNorm`. Confirm the output shape equals the input shape (that's what lets layers stack).
3. Feed a batch with padding and add a **padding mask** so `[PAD]` positions get zero attention.

In [ ]:
# Your turn: sketch an EncoderBlock here
class EncoderBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, d_ff=128):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    def forward(self, x, mask=None):
        a, _ = self.attn(x, x, x, mask)
        x = self.norm1(x + a)                 # residual + norm
        x = self.norm2(x + self.ff(x))        # residual + norm
        return x

blk = EncoderBlock()
z = blk(torch.randn(1, 5, 32))
print("EncoderBlock keeps shape:", z.shape, "  <- stack me N times!")